## 03 · E5 + bge-reranker + Qwen3 ile ajan tabanlı RAG (Colab · T4 GPU)

1. **Getirme:** `intfloat/multilingual-e5-base` + BM25 hibrit, 10 aday
2. **Yeniden sıralama:** `BAAI/bge-reranker-v2-m3` (cross-encoder)
3. **Karar:** en iyi skor eşiğin altındaysa sorgu model tarafından yeniden yazılır (en fazla 2 kez); yine bulunamazsa sistem *bilgi bulunamadı* der
4. **Yanıt:** `Qwen/Qwen3-4B-Instruct-2507`, yalnız verilen parçalardan ve kaynak adıyla

> Çalışma zamanı: *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("agentic-rag-turkish-docs").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/agentic-rag-turkish-docs.git
if IN_COLAB and Path("agentic-rag-turkish-docs").exists():
    !git -C agentic-rag-turkish-docs pull -q
    %cd agentic-rag-turkish-docs
    !pip -q install -r requirements-gpu.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import logging, time, warnings
warnings.filterwarnings("ignore")
logging.getLogger("faiss").setLevel(logging.WARNING)

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from rag_utils import load_chunks, E5Embeddings

chunks = load_chunks(chunk_size=350, chunk_overlap=50)
t0 = time.time()
vs = FAISS.from_documents(chunks, E5Embeddings(device="cuda"))
dense10 = vs.as_retriever(search_kwargs={"k": 10})
bm25_10 = BM25Retriever.from_documents(chunks, k=10)
hybrid10 = EnsembleRetriever(retrievers=[bm25_10, dense10], weights=[0.5, 0.5])
print(f"{len(chunks)} parça, vektör boyutu {vs.index.d}, indeksleme {time.time() - t0:.1f} sn")

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)

def search(query, k=10):
    cands = hybrid10.invoke(query)[:k]
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return sorted(zip(cands, map(float, scores)), key=lambda x: -x[1])

for d, s in search("Aday kaydedilmek istemezse ne olur?")[:3]:
    print(f"{s:.3f}  {d.metadata['kaynak']:<30} {d.metadata.get('bolum')}")

### Getirme değerlendirmesi (12 soru)

In [ ]:
import pandas as pd
from rag_eval import QUESTIONS, UNANSWERABLE, retrieval_metrics

def src(docs):
    return [d.metadata["kaynak"] for d in docs]

sonuc = {
    "BM25": retrieval_metrics(lambda q: src(bm25_10.invoke(q))),
    "E5 (FAISS)": retrieval_metrics(lambda q: src(dense10.invoke(q))),
    "Hibrit": retrieval_metrics(lambda q: src(hybrid10.invoke(q))),
    "Hibrit + reranker": retrieval_metrics(lambda q: src(d for d, _ in search(q))),
}
pd.DataFrame(sonuc).T.round(3)

### Yanıtı olmayan sorular ve eşik
Reranker skoru bir doğruluk olasılığı değildir; ama yanıtı belgede olan ve olmayan soruların en iyi skorları karşılaştırılarak bir *yeniden arama* eşiği seçilebilir. Küçük küme olduğu için eşik gerçek kullanımda ayrı bir doğrulama kümesiyle yeniden belirlenmeli.

In [ ]:
best = lambda q: search(q)[0][1]
cevapli = [best(q) for q, _ in QUESTIONS]
cevapsiz = [best(q) for q in UNANSWERABLE]
print(f"yanıtı olan sorular : min {min(cevapli):.3f}  medyan {sorted(cevapli)[len(cevapli)//2]:.3f}")
print(f"yanıtı olmayan      : max {max(cevapsiz):.3f}  -> {[round(s, 3) for s in cevapsiz]}")
ESIK = round((min(cevapli) + max(cevapsiz)) / 2, 3) if min(cevapli) > max(cevapsiz) else 0.5
print("seçilen eşik:", ESIK)

### Yanıt üreten model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from agentic import agentic_answer, build_context

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
print(f"GPU belleği: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

def chat(system, user, max_new_tokens=300):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def answer(question, hits):
    return chat("Yalnızca verilen belge parçalarına dayanarak Türkçe ve kısa yanıt ver. "
                "Her bilginin sonunda köşeli parantez içinde kaynak dosya adını yaz. "
                "Parçalarda yanıt yoksa bunu açıkça söyle.",
                f"Belge parçaları:\n{build_context(hits)}\n\nSoru: {question}")

def rewrite(question, previous):
    return chat("Bir şirket içi belge arama sistemi için sorguyu yeniden yaz. Eş anlamlı ve daha genel "
                "terimler kullan. Yalnızca yeni sorguyu tek satır olarak döndür.",
                f"Asıl soru: {question}\nÖnceki sorgu: {previous}", max_new_tokens=40)

### Ajan döngüsü

In [ ]:
def calistir(soru):
    t0 = time.time()
    r = agentic_answer(soru, search, answer, rewrite, threshold=ESIK, max_rewrites=2)
    print(f"SORU: {soru}")
    for adim in r["iz"]:
        print(f"  deneme {adim['deneme']}: skor={adim['en_iyi_skor']:.3f}  sorgu=\"{adim['sorgu']}\"  kaynaklar={adim['kaynaklar']}")
    print(f"YANIT ({time.time() - t0:.1f} sn): {r['yanit']}\n")

calistir("Mülakat kayıtlarını ne kadar süre tutuyoruz, transkriptler için süre farklı mı?")
calistir("İzin günlerim yılı devrederse yanar mı?")
calistir("Şirketin yemek kartı limiti ne kadar?")